In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pickle

### Functions

In [ ]:
# plot feature
def plot_feature(ser_feature, col):
    # get min/max
    flt_min = ser_feature.min()
    flt_max = ser_feature.max()
    # show distribution
    fig, ax = plt.subplots(figsize=(5,3))
    ax.set_title(f'Distribution of {col} (N = {len(ser_feature)}; Min = {flt_min:0.2f}; Max = {flt_max:0.2f})')
    sns.distplot(list(ser_feature), ax=ax)
    plt.show()

### Constants

In [ ]:
list_cols_check = [
    'payment__app',
    'fltgrossmonthly__income_sum',
    'ENG-payment_to_income',
    'amtfinanced__app',
    'bookvalue__app',
    'ENG-loan_to_value',
    'vehicleyear__app',
    'ENG-dealership_age',
    'intterm__app',
]

### AD data

In [ ]:
list_cols = [
    'uniqueid',
    'bitfunded__app',
]

str_filename = 'df_raw.gzip'
str_uri = f's3://20231010-gen-xii/01_ad/01_data_prep/01_data_collection/output/{str_filename}'
df_ad = pd.read_parquet(str_uri, columns=list_cols)

# subset
df_ad = df_ad[df_ad['bitfunded__app'] == 1]

# show
df_ad

In [ ]:
ser_val_counts = df_ad['uniqueid'].value_counts()
ser_val_counts.max()

In [ ]:
ser_val_counts

### Load target

In [ ]:
str_uri = 's3://20231010-gen-xii/ad_hoc/target_pd/df_target_pd.csv'
df_target = pd.read_csv(str_uri)

df_target

In [ ]:
ser_val_counts = df_target['uniqueid__app'].value_counts()
ser_val_counts.max()

In [ ]:
ser_val_counts

In [ ]:
df_target.isnull().mean()

In [ ]:
# plots
for col in list_cols_check:
    try:
        plot_feature(ser_feature=df_target[col], col=col)
    except KeyError as e:
        print(f'{e} not in data')

In [ ]:
del df_target

### Raw data

In [ ]:
str_uri = 's3://20231010-gen-xii/02_pricing_pd/01_data_prep/05_leaky_features/04_write_dfs/df_valid_noleaks.gzip'
df_raw = pd.read_parquet(str_uri)

df_raw

In [ ]:
ser_val_counts = df_raw['uniqueid'].value_counts()
ser_val_counts.max()

In [ ]:
ser_val_counts

In [ ]:
list_cols_tmp = [col for col in list_cols_check if col in list(df_raw.columns)]
df_raw[list_cols_tmp].isnull().mean()

In [ ]:
# plots
for col in list_cols_check:
    try:
        plot_feature(ser_feature=df_raw[col], col=col)
    except KeyError as e:
        print(f'{e} not in data')

### Preprocessed data

In [ ]:
str_uri = 's3://20231010-gen-xii/02_pricing_pd/02_model/00_preprocessing/02_make_dfs/df_valid_noleaks_pre.gzip'
df_clean = pd.read_parquet(str_uri)

df_clean

In [ ]:
ser_val_counts = df_clean['uniqueid'].value_counts()
ser_val_counts.max()

In [ ]:
ser_val_counts

In [ ]:
list_cols_tmp = [col for col in list_cols_check if col in list(df_clean.columns)]
df_clean[list_cols_tmp].isnull().mean()

In [ ]:
# plots
for col in list_cols_check:
    try:
        plot_feature(ser_feature=df_clean[col], col=col)
    except KeyError as e:
        print(f'{e} not in data')

### Preprocess

In [ ]:
cls_model_preprocessing = pickle.load(open('cls_model_preprocessing.pkl', 'rb'))
list_transformers = cls_model_preprocessing.list_transformers
list_transformers

In [ ]:
# iterate through transformers
for a in range(len(list_transformers)):
    list_transformers_tmp = list_transformers.copy()[:a]
    # make copy
    df_raw_copy = df_raw.copy()
    for transformer in list_transformers_tmp:
        # transform df_raw_copy
        df_raw_copy = transformer.transform(df_raw_copy)
        # plots
        for col in list_cols_check:
            try:
                plot_feature(ser_feature=df_raw_copy[col], col=col)
            except KeyError as e:
                print(f'{e} not in data')